In [ ]:
# ============================================================
# CELL 1: SETUP (Chạy 1 lần)
# ============================================================

# Mở khóa và kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# [FIX OOM T4] Đặt TRƯỚC khi import torch - giảm phân mảnh VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Tắt torch.compile (nguyên nhân gây tốn thêm rất nhiều VRAM khi compile trên Colab/Linux)
os.environ["TORCH_COMPILE_DISABLE"] = "1"

# Khai báo các thư mục lưu trữ bền vững
DRIVE_PATH = '/content/drive/MyDrive/MangaTranslator_Data'
MODELS_DIR = f'{DRIVE_PATH}/models'
FONTS_DIR = f'{DRIVE_PATH}/fonts'
PIP_CACHE_DIR = f'{DRIVE_PATH}/pip_cache'
CONFIG_DIR = f'{DRIVE_PATH}/config'
OUTPUT_DIR = f'{DRIVE_PATH}/output'  # <- Thư mục truyện sẽ tự động xuất ra đây

# Tạo các thư mục trên Drive (nếu chưa có)
for d in [MODELS_DIR, FONTS_DIR, PIP_CACHE_DIR, CONFIG_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Clone project
%cd /content
!rm -rf MangaTranslator
!git clone -q https://github.com/meangrinch/MangaTranslator.git
%cd MangaTranslator

# 1. Symlink models, fonts và ĐẶC BIỆT LÀ OUTPUT
!rm -rf models fonts output
!ln -s "{MODELS_DIR}" ./models
!ln -s "{FONTS_DIR}" ./fonts
!ln -s "{OUTPUT_DIR}" ./output  # <- Truyện dịch xong tự lưu vào Drive

# 2. Symlink thư mục cấu hình (Lưu API Key)
!rm -rf /root/MangaTranslator
!ln -s "{CONFIG_DIR}" /root/MangaTranslator

# 3. Cài đặt thư viện với Cache siêu tốc
!pip install -q --cache-dir="{PIP_CACHE_DIR}" -r requirements.txt

# 4. Vá lỗi Gradio Public Link
!sed -i 's/app.launch(inbrowser=args.open_browser, server_port=args.port, show_error=True)/app.launch(inbrowser=args.open_browser, server_port=args.port, show_error=True, share=True)/g' app.py

# 5. [FIX OOM T4] Giải phóng VRAM (YOLO/SAM/Upscaler) trước khi Flux chạy
import textwrap

target_file = 'core/outside_text_processor.py'
with open(target_file, 'r') as f:
    content = f.read()

old_line = 'log_message("Inpainting outside text regions...", verbose=verbose)'
new_block = textwrap.dedent('''\
log_message("Inpainting outside text regions...", verbose=verbose)

        # [Colab T4 VRAM Fix] Giải phóng GPU trước khi load Flux
        _mgr = get_model_manager()
        _mgr.unload_ocr_models(verbose=verbose)
        _mgr.unload_upscale_models(verbose=verbose)
        _mgr.clear_cache()
        import gc as _gc; _gc.collect()
        log_message("[Colab T4] Freed VRAM for Flux inpainting", always_print=True)''')

if old_line in content:
    content = content.replace(old_line, new_block, 1)
    with open(target_file, 'w') as f:
        f.write(content)
    print('✅ Patched outside_text_processor.py - VRAM cleanup')

# 6. [FIX OOM T4] Tắt torch.compile trong sdnq 
mm_file = 'core/ml/model_manager.py'
with open(mm_file, 'r') as f:
    mm_content = f.read()
    
mm_content = mm_content.replace(
    'if triton_is_available and has_gpu:',
    'if False:  # [Colab T4 VRAM Fix] Disabled triton/torch.compile to save VRAM'
)
with open(mm_file, 'w') as f:
    f.write(mm_content)
print('✅ Patched model_manager.py - Disabled torch.compile')

# 7. [FIX OOM T4] Bắt buộc giải phóng Text Encoder (T5) khỏi GPU cho TOÀN BỘ classes Flux
inpaint_file = 'core/image/inpainting.py'
with open(inpaint_file, 'r') as f:
    inpaint_content = f.read()

old_embeds_code = '''\
            self._prompt_embeds_cpu = _prompt_value_to_cpu(prompt_embeds)
            self._pooled_prompt_embeds_cpu = _prompt_value_to_cpu(pooled_prompt_embeds)'''

new_embeds_code = '''\
            self._prompt_embeds_cpu = _prompt_value_to_cpu(prompt_embeds)
            self._pooled_prompt_embeds_cpu = _prompt_value_to_cpu(pooled_prompt_embeds)
            
            # [Colab T4 VRAM Fix] Force unload text encoders to CPU after encoding
            if hasattr(self.pipeline, "text_encoder") and self.pipeline.text_encoder is not None:
                self.pipeline.text_encoder.to("cpu")
            if hasattr(self.pipeline, "text_encoder_2") and self.pipeline.text_encoder_2 is not None:
                self.pipeline.text_encoder_2.to("cpu")
            empty_cache(device)
            log_message("[Colab T4] Forced text encoders to CPU to free 9.4GB VRAM", always_print=True)'''

if old_embeds_code in inpaint_content:
    # Sửa lỗi trước đó: dùng replace toàn bộ thay vì replace(..., 1) để đảm bảo FluxKleinInpainter cũng được patch
    inpaint_content = inpaint_content.replace(old_embeds_code, new_embeds_code)
    with open(inpaint_file, 'w') as f:
        f.write(inpaint_content)
    print('✅ Patched inpainting.py - Forced T5 Text Encoder to CPU for all Flux models')
else:
    print('⚠️ Could not patch inpainting.py - block not found')

print("✅ Môi trường hoàn hảo! Truyện dịch xong sẽ tự động bay vào Google Drive của bạn.")

In [ ]:
# ============================================================
# CELL 2: CHẠY WEB (Chạy mỗi khi cần)
# ============================================================
%cd /content/MangaTranslator
!python app.py